In [35]:
import pandas as pd
import json
from scripts.generate_finetuning_data import get_client, generate_response

In [30]:
system_prompt = """You are a full duplex conversational agent. Your job is to respond to the user, at the appropriate time and with the appropriate content. This manifests in the form of generating special FSM control tokens, as well as the response content. The FSM has two states - SPEAK and LISTEN. The control tokens are - [S.SPEAK] (start speaking) for LISTEN -> SPEAK, [C.SPEAK] (continue speaking) SPEAK -> SPEAK, [S.LISTEN] (start listen) SPEAK -> LISTEN, [C.LISTEN] (continue listen) LISTEN -> LISTEN.

When the user is speaking, the FSM is in the LISTEN state. The FSM should stay in the LISTEN state, if the user is not finished speaking, or has more content to add. 

When the FSM is in the SPEAK state, it should only transition to the LISTEN when the user has a different question to ask, the user denies or provides a negative response to the current system utterance or the agent has completed its response. The FSM should continue to be in the SPEAK state in the following cases. 
1. When there is any external noise, or disturbance, ignore and continue speaking.
2. When the user acknowledges, or provides acknowledgments/confirmations (back-channeling), continue to be in the SPEAK state.

Given the conversational history and the last words/phrases the user uttered, your job is to generate the control token. If the control token results in the FSM to end up in the SPEAK state, you should also generate the content to be spoken, which well then be relayed to a TTS system.

Sample Conversation Format:
<Conversation History>
User: ...
Agent: ...
User: ...
.
.
User: ...
Agent: [control token] ...
"""

In [31]:
import re

def extract_content(text):
    user_pattern = r"User Content: (.*?)(?:\n|$)"
    assistant_pattern = r"Assistant Content: (.*?)(?:\n|$)"
    
    user_content = re.findall(user_pattern, text)
    assistant_content = re.findall(assistant_pattern, text)
    
    return user_content, assistant_content

In [32]:
conversation = """1
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: My top travel destination would be Japan because of its rich culture, beautiful landscapes, and delicious food.
User Analysis: Isn't interrupted.
User Content: Cool, I've never thought about Japan. Did you mention food because of sushi?
--
Round: 2
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Yes, sushi is part of it, but there's also ramen, tempura, and many regional specialties.
User Analysis: The user's statement contains an obvious factual error.
User Content: But isn't sushi originally from China?
--
Round: 3
Assistant Analysis: The assistant should politely correct the error and provide useful information.
Assistant Content: Actually, while China did influence the development of sushi, it's commonly associated with Japan.
User Analysis: Interrupts the assistant with a follow-up question on the same topic.
User Content: Got it. What about the other food items you mentioned, are they also influenced by China <NOT_FINISHED>
--
Round: 4
Assistant Analysis: The assistant should answer the new question. 
Assistant Content: Ramen has Chinese origins, but tempura was influenced by Portuguese cuisine. Japanese cuisine is quite diverse!
User Analysis: Expresses denial or dissatisfaction with the response in 15 words or less.
User Content: Really? I didn't know that. I thought tempura was purely Japanese <NOT_FINISHED>
--
Round: 5
Assistant Analysis: The assistant should stop its response and address the user’s concerns.
Assistant Content: It's a common misconception, but yes, tempura was influenced by Portuguese traders in the 16th century.
User Analysis: Isn't interrupted.
User Content: That's really interesting. What other misconceptions are there about Japanese food, do you know?
--
Round: 6
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: One misconception is that all Japanese food is healthy, but like all cuisines, it has a mix of both healthy and indulgent foods.
User Analysis: Incomplete question but contains enough information for the AI assistant to respond.
User Content: Indulgent foods, like what?
--
Round: 7
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Examples are tonkatsu, a breaded and deep-fried pork cutlet, and yakitori, which can be fatty depending on the cut of meat used.
User Analysis: Isn't interrupted.
User Content: Sounds delicious! I'll definitely need to try those if I ever visit.
--
Round: 8
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Absolutely! Trying the local cuisine is always a highlight of any trip.
User Analysis: Changes the topic after receiving the desired information in 30 words or less.
User Content: Agreed. By the way, do you have any book recommendations? I'd love to read something during the flight <NOT_FINISHED>
--
Round: 9
Assistant Analysis: The assistant should respond to the new topic.
Assistant Content: For a direct connection to Japan, I'd recommend "Memoirs of a Geisha" or "Norwegian Wood" by Haruki Murakami.
User Analysis: Full question.
User Content: I've read Norwegian Wood already! Does "Memoirs of a Geisha" have a lot of cultural insights?
--
Round: 10
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Yes, it offers an engaging look into the secretive world of geishas in 20th century Japan.
User Analysis: Isn't interrupted.
User Content: Sounds interesting! I'll check that out. Thanks!
--
Round: 11
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: You're welcome! I think you'll enjoy it. Anything else you would like to know?
User Analysis: Isn't interrupted.
User Content: No, that's all for now. Thank you for the information!
--
Round: 12
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: My pleasure! Don't hesitate to ask if you have more questions in the future.
User Analysis: The user’s input is severely truncated. The AI assistant cannot answer the question due to insufficient information.
User Content: Actually, I do have another ques <NOT_FINISHED>
--
Round: 13
Assistant Analysis: The assistant cannot answer the question due to insufficient information and should ask for more details.
Assistant Content: I'm sorry, it seems your question got cut off. Could you please repeat it?
User Analysis: Isn't interrupted.
User Content: Oh yeah, sorry about that. I was asking if you have any movie recommendations as well?
--
Round: 14
Assistant Analysis: The assistant will not be interrupted in this round.
Assistant Content: Of course. A good Japanese movie is "Spirited Away," an acclaimed animation by Hayao Miyazaki.
User Analysis: Unrelated content is recorded due to background noise, in 15 words or less.
User Content: I've heard about that one. And oh, someone's at the door <NOT_FINISHED>
--
Round: 15
Assistant Analysis: The assistant should continue its response unaffected but transition naturally, using filler words.
Assistant Content: No problem at all, hope it's not too much of an interruption. I'd also recommend "Your Name," another popular Japanese film.
User Analysis: Expresses satisfaction with the response in 3 words or less.
User Content: Good choices.
--
Round: 16
Assistant Analysis: The assistant should continue its response unaffected but transition naturally, using filler words.
Assistant Content: I'm glad you think so! You'll definitely enjoy both films.
User Analysis: Isn't interrupted.
User Content: Thanks, I'll add them to the list.
--
Round: 17
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: Great! I'm sure you'll have a wonderful trip to Japan and enjoy these recommendations.
User Analysis: Isn't interrupted.
User Content: Thanks again, you've been really helpful.
--
Round: 18
Assistant Analysis: The assistant will not be interrupted.
Assistant Content: You're very welcome! It was a pleasure assisting you. Safe travels!"""

In [5]:
topic_question = 'Which country is your top travel destination and why?'

In [6]:
user_content, assistant_content = extract_content(conversation)

In [7]:
user_content = [topic_question] + user_content
user_content = [f"User: {x}" for x in user_content]
assistant_content = [f"Assistant: {x}" for x in assistant_content]

In [8]:
def interleave_lists(list1, list2):
    # Use itertools.zip_longest to pair elements, filling shorter list with None
    from itertools import zip_longest
    
    # Interleave the lists and flatten the result
    interleaved = [item for pair in zip_longest(list1, list2) for item in pair if item is not None]
    
    return interleaved

In [37]:
print("\n".join(interleave_lists(user_content[:], assistant_content[:])))

User: Which country is your top travel destination and why?
Assistant: My top travel destination would be Japan because of its rich culture, beautiful landscapes, and delicious food.
User: Cool, I've never thought about Japan. Did you mention food because of sushi?
Assistant: Yes, sushi is part of it, but there's also ramen, tempura, and many regional specialties.
User: But isn't sushi originally from China?
Assistant: Actually, while China did influence the development of sushi, it's commonly associated with Japan.
User: Got it. What about the other food items you mentioned, are they also influenced by China <NOT_FINISHED>
Assistant: Ramen has Chinese origins, but tempura was influenced by Portuguese cuisine. Japanese cuisine is quite diverse!
User: Really? I didn't know that. I thought tempura was purely Japanese <NOT_FINISHED>
Assistant: It's a common misconception, but yes, tempura was influenced by Portuguese traders in the 16th century.
User: That's really interesting. What other

In [34]:
input_data = pd.read_csv('/Users/soham/Desktop/CyborgVoice/generated_gpt4_data.csv')

In [11]:
client = get_client()

In [17]:
print(generate_response(client, prompt=input_prompt, system_prompt=system_prompt))

Assistant: [C.SPEAK] It's a common misconception, but as I mentioned, tempura was influenced by the Portuguese, specifically by a cooking technique they introduced to Japan in the 16th century. And as for ramen, while it was indeed influenced by Chinese wheat noodles, the Japanese have made it their own with unique variations and flavors.


In [26]:
print(generate_response(client, prompt=input_prompt, system_prompt=system_prompt))

Assistant: [S.SPEAK] Absolutely! For non-fiction, I would recommend "The Book of Five Rings" by Miyamoto Musashi. For fiction, "Norwegian Wood" by Haruki Murakami is a great choice. Both provide interesting insights into Japanese culture and society.


In [ ]:
{'num_rounds': 18, 'denial_round': 5, 'inquiry_round': 4, 'topic_change_round': 9, 'noise_round': 15, 'acknowledgment_round': 16, 'lack_round': 16, 'complete_round': 10, 'error_round': 3, 'first_question_topic': 'Which country is your top travel destination and why?', 'response_word_count': 12, 'interrupted_response_word_count': 14}

In [43]:
from tqdm.auto import tqdm

In [62]:
dataset_preds = {}
conditions = {
    "[S.SPEAK]": ['lack_round', 'complete_round', 'error_round'],
    "[C.SPEAK]": ['noise_round', 'acknowledgment_round'],
    "[S.LISTEN]": ['denial_round', 'inquiry_round', 'topic_change_round']
}
for i, row in tqdm(input_data.iterrows(), total=len(input_data)):
    dataset_preds[i] = {}
    kwargs = eval(row['kwargs'])
    conversation_raw = row['response']
    topic_question = kwargs['first_question_topic']
    if not f"Round: {kwargs['num_rounds']}" in conversation_raw:
        continue
    
    
    user_content, assistant_content = extract_content(conversation_raw)
    user_content = [topic_question] + user_content
    user_content = [f"User: {x}" for x in user_content]
    assistant_content = [f"Assistant: {x}" for x in assistant_content]
    for condition, round_types in conditions.items():
        dataset_preds[i][condition] = {}
        for round_type in round_types:
            ind = kwargs[round_type]
            input_prompt = "\n".join(interleave_lists(user_content[:ind], assistant_content[:ind - 1]))
            resp = generate_response(client, prompt=input_prompt, system_prompt=system_prompt)
            success = condition in resp
            dataset_preds[i][condition][round_type] = {
                "input_prompt": input_prompt,
                "resp": resp,
                "success": success,
                "round_num": ind
            }
    

100%|██████████| 10/10 [03:11<00:00, 19.13s/it]


In [64]:
with open("prompt_v0_results.json", 'w') as f:
    json.dump(dataset_preds, f)